[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/01_Multimodal_Foundations/01_what_is_multimodal/01_what_is_multimodal.ipynb)

# 01. What is Multimodal Learning?

**You already know:** Self-attention, Multi-head attention, Cross-attention, Transformers

**This notebook covers:**
- What "multimodal" means and why it matters
- The landscape of multimodal models (CLIP, LLaVA, Flamingo, etc.)
- How different modalities are represented
- The key challenge: aligning different representation spaces
- **Hands-on:** Build a Mini-CLIP model from scratch

**Runtime:** ~5 minutes on CPU | ~2 minutes on Colab GPU

---

In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    # Step 1: Clone the repository
    if not os.path.exists(REPO_DIR):
        print("📥 Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("✅ Repository already cloned")

    # Step 2: Install dependencies (Colab-safe subset)
    print("📦 Installing dependencies...")
    !pip install -q torch torchvision torchaudio
    !pip install -q transformers datasets accelerate peft
    !pip install -q matplotlib seaborn numpy pandas scikit-learn tqdm
    !pip install -q einops timm sentencepiece tokenizers safetensors
    !pip install -q open-clip-torch gradio onnx onnxruntime
    !pip install -q ipywidgets pillow

    # Step 3: Set working directory
    MODULE_DIR = f"{REPO_DIR}/01_Multimodal_Foundations/01_what_is_multimodal"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    # Step 4: Add project root to Python path
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"\n✅ Colab setup complete!")
    print(f"   Working directory: {os.getcwd()}")

    # Show GPU info if available
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"   GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    else:
        print("   Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

# Import project utilities (visualization helpers)
try:
    from utils.visualization import *
    from utils.helpers import count_parameters
    set_style()
    print("✅ Utilities loaded successfully")
except ImportError as e:
    print(f"⚠️  Could not import utils ({e})")
    print("   Defining inline fallbacks — notebook will still work fine.")

    import seaborn as sns

    def set_style():
        plt.rcParams.update({
            'figure.figsize': (12, 6), 'figure.dpi': 100,
            'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12,
            'axes.grid': True, 'grid.alpha': 0.3, 'figure.facecolor': 'white',
        })
        sns.set_palette("husl")

    def draw_architecture_block(ax, x, y, w, h, label, color='#4ECDC4', fontsize=10):
        box = FancyBboxPatch((x - w/2, y - h/2), w, h,
            boxstyle="round,pad=0.1", facecolor=color, edgecolor='#2C3E50',
            linewidth=1.5, alpha=0.85)
        ax.add_patch(box)
        ax.text(x, y, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color='white')
        return box

    def draw_arrow(ax, start, end, color='#2C3E50', style='->', lw=1.5):
        ax.annotate('', xy=end, xytext=start,
                    arrowprops=dict(arrowstyle=style, color=color, lw=lw))

    def plot_multimodal_overview():
        set_style()
        fig, ax = plt.subplots(1, 1, figsize=(14, 8))
        ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')
        ax.set_title('Multimodal Architecture Overview', fontsize=18, fontweight='bold', pad=20)
        colors = {'image': '#E74C3C', 'text': '#3498DB', 'audio': '#2ECC71',
                  'fusion': '#9B59B6', 'output': '#F39C12', 'encoder': '#1ABC9C'}
        draw_architecture_block(ax, 2, 6.5, 2.5, 1, 'Image\n(pixels)', colors['image'])
        draw_architecture_block(ax, 7, 6.5, 2.5, 1, 'Text\n(tokens)', colors['text'])
        draw_architecture_block(ax, 12, 6.5, 2.5, 1, 'Audio\n(spectrogram)', colors['audio'])
        draw_architecture_block(ax, 2, 4.5, 2.5, 1, 'Vision Encoder\n(ViT / CNN)', colors['encoder'])
        draw_architecture_block(ax, 7, 4.5, 2.5, 1, 'Text Encoder\n(BERT / GPT)', colors['encoder'])
        draw_architecture_block(ax, 12, 4.5, 2.5, 1, 'Audio Encoder\n(Whisper)', colors['encoder'])
        for x in [2, 7, 12]:
            draw_arrow(ax, (x, 6.0), (x, 5.1))
        draw_architecture_block(ax, 7, 2.5, 8, 1.2,
            'Fusion Layer\n(Cross-Attention / Concatenation / Gating)', colors['fusion'], fontsize=12)
        for x in [2, 7, 12]:
            draw_arrow(ax, (x, 4.0), (x if x == 7 else (4.5 if x == 2 else 9.5), 3.2))
        draw_architecture_block(ax, 7, 0.8, 4, 0.9,
            'Task Output (Classification / Generation)', colors['output'], fontsize=10)
        draw_arrow(ax, (7, 1.9), (7, 1.35))
        plt.tight_layout()
        return fig

    def plot_attention_heatmap(attention_weights, x_labels=None, y_labels=None, title='Attention Weights'):
        set_style()
        if isinstance(attention_weights, torch.Tensor):
            attention_weights = attention_weights.detach().cpu().numpy()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(attention_weights, annot=True, fmt='.3f', cmap='YlOrRd',
                    xticklabels=x_labels, yticklabels=y_labels,
                    ax=ax, cbar_kws={'label': 'Attention Weight'})
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_xlabel('Keys'); ax.set_ylabel('Queries')
        plt.tight_layout()
        return fig

    def count_parameters(model, print_table=True):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        frozen = total - trainable
        if print_table:
            print(f"{'='*50}")
            print(f"{'Parameter Summary':^50}")
            print(f"{'='*50}")
            print(f"  Total parameters:     {total:>12,}")
            print(f"  Trainable parameters: {trainable:>12,}")
            print(f"  Frozen parameters:    {frozen:>12,}")
            print(f"  Trainable %:          {trainable/total*100:>11.2f}%")
            print(f"{'='*50}")
        return {'total': total, 'trainable': trainable, 'frozen': frozen}

    set_style()
    print("✅ Inline utilities ready")

print(f"\nPyTorch version: {torch.__version__}")
print(f"Device: {'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

## 1. The Multimodal Landscape

**Single-modal:** One input type (text OR image OR audio)  
**Multi-modal:** Multiple input types working together

| Model | Year | Modalities | What it does | Open Source? |
|-------|------|-----------|-------|------------|
| **CLIP** | 2021 | Image + Text | Matches images to text descriptions | ✅ Yes |
| **LLaVA** | 2023 | Image + Text | Answers questions about images | ✅ Yes |
| **Flamingo** | 2022 | Image + Text | Few-shot visual reasoning | ❌ No |
| **Whisper** | 2022 | Audio → Text | Speech to text | ✅ Yes |
| **ImageBind** | 2023 | 6 modalities | Unified embedding space | ✅ Yes |
| **GPT-4V** | 2023 | Image + Text | General multimodal reasoning | ❌ No |
| **Gemini** | 2023 | Native multi-modal | Reasoning across modalities | ❌ No |

> **Key insight:** Almost all of these share the same recipe — **separate encoders + shared embedding space + alignment training**. We'll build this pattern from scratch below.

## Key Paper Figures

The following diagrams recreate iconic figures from foundational multimodal papers.

![CLIP Architecture — Radford et al. (2021)](../../assets/paper_figures/clip_overview.png)
*Source: Radford et al. (2021) — Learning Transferable Visual Models From Natural Language Supervision*

![CLIP Architecture — Radford et al. (2021)](../assets/paper_figure_clip.png)

*Source: Radford et al. (2021) — "Learning Transferable Visual Models From Natural Language Supervision" — [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)*

## 📚 Key Multimodal Papers: A Timeline

Understanding the evolution of multimodal AI requires familiarity with these landmark papers:

### 2020-2021: Foundation Models

| Paper | Key Innovation | Modalities | Math Insight |
|-------|---------------|------------|--------------|
| **ViLBERT** (Lu et al., 2019) | Co-attention between dual streams | Image + Text | Cross-modal attention: $Q_v K_t^T / \sqrt{d}$ |
| **CLIP** (Radford et al., 2021) | Contrastive image-text pretraining at scale | Image + Text | $\mathcal{L} = -\frac{1}{N}\sum_i \log \frac{e^{s_{ii}/\tau}}{\sum_j e^{s_{ij}/\tau}}$ |
| **ALIGN** (Jia et al., 2021) | Scale > curation (1.8B noisy pairs) | Image + Text | Same contrastive loss as CLIP, but trained on noisy data |
| **DALL·E** (Ramesh et al., 2021) | Text-to-image generation via dVAE + Transformer | Text → Image | $p(x, y) = \prod_i p(x_i \mid x_{1:i-1}, y)$ autoregressively |

### 2022: Scaling & Multi-task

| Paper | Key Innovation | Modalities | Math Insight |
|-------|---------------|------------|--------------|
| **Flamingo** (Alayrac et al., 2022) | Few-shot multimodal learning with gated cross-attention | Image + Text | Gated x-attn: $y = \tanh(\alpha) \cdot \text{xattn}(x) + x$, $\alpha$ init to 0 |
| **BLIP** (Li et al., 2022) | Unified ITC+ITM+LM with CapFilt bootstrapping | Image + Text | 3 objectives, shared encoder, filter noisy web data |
| **CoCa** (Yu et al., 2022) | Contrastive + Captioning in one model | Image + Text | $\mathcal{L} = \mathcal{L}_{ITC} + \lambda \mathcal{L}_{Cap}$ |
| **CLAP** (Elizalde et al., 2023) | CLIP for audio — contrastive audio-text | Audio + Text | Same InfoNCE framework, different encoder (HTSAT/PANN) |

### 2023-2024: LLM-based Multimodal

| Paper | Key Innovation | Modalities | Math Insight |
|-------|---------------|------------|--------------|
| **LLaVA** (Liu et al., 2023) | Visual instruction tuning with simple MLP projector | Image + Text + Instructions | 2-stage: alignment → instruction tuning |
| **GPT-4V** (OpenAI, 2023) | Multimodal GPT at scale | Image + Text | Undisclosed architecture |
| **Gemini** (Google, 2023) | Natively multimodal from pretraining | Image + Text + Audio + Video | Trained jointly on all modalities |
| **ImageBind** (Girdhar et al., 2023) | 6 modalities, 1 embedding space | Image, Text, Audio, Depth, IMU, Thermal | Images as binding modality |

![Flamingo Architecture — Alayrac et al. (2022)](../assets/paper_figure_flamingo_dalle.png)

*Source: Alayrac et al. (2022) — "Flamingo: a Visual Language Model for Few-Shot Learning" — [arXiv:2204.14198](https://arxiv.org/abs/2204.14198)*

*See also: Ramesh et al. (2021) — DALL·E — [arXiv:2102.12092](https://arxiv.org/abs/2102.12092)*

## 🔊 CLAP: Extending Contrastive Learning to Audio

**Paper:** Elizalde et al. (2023) — *CLAP: Learning Audio Concepts from Natural Language Supervision* — [arXiv:2206.04769](https://arxiv.org/abs/2206.04769)

CLAP applies the CLIP framework to audio-text pairs. Key differences from CLIP:

### Architecture Comparison

| Component | CLIP | CLAP |
|-----------|------|------|
| **Visual/Audio Encoder** | ViT or ResNet | HTSAT (audio spectrogram transformer) or PANN (CNN) |
| **Text Encoder** | Transformer (63M params) | RoBERTa or BERT |
| **Input preprocessing** | Resize + normalize image | Audio → Mel spectrogram → patches |
| **Training data** | 400M image-text pairs (WIT) | 128K audio-text pairs (AudioCaps, Clotho, LAION-Audio) |
| **Zero-shot task** | Image classification | Audio classification, sound event detection |

### CLAP Loss (identical structure to CLIP)

$$
\mathcal{L}_{CLAP} = \frac{1}{2}\left(\mathcal{L}_{a \rightarrow t} + \mathcal{L}_{t \rightarrow a}\right)
$$

where:

$$
\mathcal{L}_{a \rightarrow t} = -\frac{1}{N}\sum_{i=1}^{N}\log\frac{\exp(\text{sim}(a_i, t_i)/\tau)}{\sum_{j=1}^{N}\exp(\text{sim}(a_i, t_j)/\tau)}
$$

### Audio Preprocessing Pipeline

```
Raw Waveform (16kHz, mono)
    ↓ STFT (window=25ms, hop=10ms)
    ↓ Mel Filterbank (64 or 128 bands)
    ↓ Log-Mel Spectrogram
    ↓ Patchify (like ViT patches)
    ↓ Audio Encoder (HTSAT / PANN)
    ↓ [CLS] embedding → Audio representation
```

### Key Results
- **ESC-50:** 93.7% zero-shot accuracy (vs. supervised baseline 95.2%)
- **UrbanSound8K:** 73.2% zero-shot
- Enables **zero-shot audio classification** — just provide text descriptions of sound classes!

In [ ]:
# Visualize the multimodal architecture overview
fig = plot_multimodal_overview()
plt.savefig('../assets/multimodal_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to assets/multimodal_overview.png")

## 2. How Different Modalities Are Represented

The core challenge: **every modality has a different native format.**
We need to convert them all into a **common representation** (embedding vectors).

| Modality | Raw Format | Encoder | Output |
|----------|-----------|---------|--------|
| **Image** | Pixels $\in \mathbb{R}^{H \times W \times 3}$ | ViT / CNN | $\mathbf{h} \in \mathbb{R}^D$ |
| **Text** | Token IDs $\in \mathbb{Z}^T$ | BERT / GPT | $\mathbf{h} \in \mathbb{R}^D$ |
| **Audio** | Waveform $\in \mathbb{R}^L$ | Whisper / wav2vec | $\mathbf{h} \in \mathbb{R}^D$ |

> Notice: all encoders output **the same shape** $\mathbf{h} \in \mathbb{R}^D$. That's the key to multimodal fusion!

In [ ]:
# Visualize: How each modality becomes a vector
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Image → Patches → Embeddings
ax = axes[0]
ax.set_title('Image → Embeddings', fontsize=14, fontweight='bold')
img = np.random.rand(8, 8, 3) * 0.5 + 0.3
ax.imshow(img, extent=[0, 4, 4, 8])

# Show patch grid
for i in range(0, 5, 2):
    ax.axhline(y=4+i, xmin=0, xmax=0.4, color='white', lw=2)
    ax.axvline(x=i*0.5, ymin=0.5, ymax=1.0, color='white', lw=2)

# Show embedding vectors
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
for i, c in enumerate(colors):
    vec = np.random.randn(1, 10) * 0.5
    ax.barh(3 - i * 0.7, vec[0], height=0.5, color=c, alpha=0.7, left=0)
    ax.text(-2.5, 3 - i * 0.7, f'patch {i+1}', fontsize=9, va='center')

ax.set_xlim(-3, 5)
ax.set_ylim(-0.5, 8.5)
ax.arrow(2, 3.8, 0, -0.5, head_width=0.2, color='black')
ax.text(2, 4.2, 'ViT / CNN', ha='center', fontsize=10, fontweight='bold')
ax.axis('off')

# Text → Tokens → Embeddings
ax = axes[1]
ax.set_title('Text → Embeddings', fontsize=14, fontweight='bold')
tokens = ['[CLS]', 'a', 'cat', 'sitting', '[SEP]']
for i, tok in enumerate(tokens):
    ax.add_patch(FancyBboxPatch((i*1.5+0.5, 6), 1.2, 0.8, 
                                boxstyle='round,pad=0.1', facecolor='#3498DB', alpha=0.7))
    ax.text(i*1.5+1.1, 6.4, tok, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

for i in range(5):
    vec = np.random.randn(1, 10) * 0.5
    ax.barh(4 - i * 0.7, vec[0], height=0.5, color='#3498DB', alpha=0.6, left=0.5)
    ax.text(-1, 4 - i * 0.7, tokens[i], fontsize=9, va='center')

ax.arrow(4, 5.8, 0, -0.5, head_width=0.2, color='black')
ax.text(4, 6.2, 'Tokenizer + Encoder', ha='center', fontsize=10, fontweight='bold')
ax.set_xlim(-2, 9)
ax.set_ylim(0.5, 8)
ax.axis('off')

# Audio → Spectrogram → Embeddings
ax = axes[2]
ax.set_title('Audio → Embeddings', fontsize=14, fontweight='bold')
spec = np.random.rand(20, 40) ** 2
ax.imshow(spec, aspect='auto', cmap='magma', extent=[0.5, 7.5, 4.5, 7.5])
ax.text(4, 7.8, 'Mel Spectrogram', ha='center', fontsize=10)

for i in range(4):
    vec = np.random.randn(1, 10) * 0.5
    ax.barh(3.5 - i * 0.7, vec[0], height=0.5, color='#2ECC71', alpha=0.6, left=0.5)
    ax.text(-1, 3.5 - i * 0.7, f'frame {i+1}', fontsize=9, va='center')

ax.arrow(4, 4.3, 0, -0.3, head_width=0.2, color='black')
ax.text(4, 4.7, 'Whisper / wav2vec', ha='center', fontsize=10, fontweight='bold')
ax.set_xlim(-2, 9)
ax.set_ylim(0.5, 8.5)
ax.axis('off')

plt.tight_layout()
plt.savefig('../assets/modality_representations.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. The Alignment Problem (Core Challenge)

**Problem:** A cat image and the text "a photo of a cat" should map to **nearby** points in embedding space.

Mathematically, we want the **cosine similarity** between matching pairs to be high:

$$\text{sim}(\mathbf{v}_{\text{cat}}, \mathbf{t}_{\text{"a cat"}}) = \frac{\mathbf{v}_{\text{cat}} \cdot \mathbf{t}_{\text{"a cat"}}}{\|\mathbf{v}_{\text{cat}}\| \cdot \|\mathbf{t}_{\text{"a cat"}}\|} \to 1$$

And similarity between non-matching pairs to be low:

$$\text{sim}(\mathbf{v}_{\text{cat}}, \mathbf{t}_{\text{"a dog"}}) \to 0$$

**Before training:** All similarities are random (~0).  
**After training:** Matching pairs cluster together!

### Cosine Similarity — Numerical Walkthrough

Let's compute similarity by hand with 4-dimensional vectors:

$$\mathbf{v} = [0.3,\; 0.8,\; 0.1,\; 0.5], \quad \mathbf{t} = [0.4,\; 0.7,\; 0.2,\; 0.6]$$

**Step 1 — Dot product:**

$$\mathbf{v} \cdot \mathbf{t} = (0.3)(0.4) + (0.8)(0.7) + (0.1)(0.2) + (0.5)(0.6) = 0.12 + 0.56 + 0.02 + 0.30 = 1.00$$

**Step 2 — L2 norms:**

$$\|\mathbf{v}\| = \sqrt{0.09 + 0.64 + 0.01 + 0.25} = \sqrt{0.99} \approx 0.995$$

$$\|\mathbf{t}\| = \sqrt{0.16 + 0.49 + 0.04 + 0.36} = \sqrt{1.05} \approx 1.025$$

**Step 3 — Cosine similarity:**

$$\text{sim}(\mathbf{v}, \mathbf{t}) = \frac{1.00}{0.995 \times 1.025} \approx \frac{1.00}{1.020} \approx 0.980$$

A score of **0.98** means these vectors point in nearly the same direction — exactly what we want for a matching image-text pair after alignment training.

### L2 Normalization — Why It Matters

Before computing cosine similarity, CLIP **L2-normalizes** both embedding vectors:

$$\hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2}, \quad \hat{\mathbf{t}} = \frac{\mathbf{t}}{\|\mathbf{t}\|_2}$$

**Three critical reasons:**

1. **Dot product = cosine similarity:** After normalization, $\hat{\mathbf{v}} \cdot \hat{\mathbf{t}} = \cos(\theta)$ — no need to compute norms separately at inference time
2. **Bounded output:** Cosine similarity is always in $[-1, 1]$, preventing any single embedding dimension from dominating the score
3. **Training stability:** Without normalization, the model can cheat by scaling embeddings larger instead of learning meaningful directions — L2 norm forces the model to learn **direction** (semantic content), not magnitude

This is why our `MiniCLIP` applies `emb / emb.norm(dim=-1, keepdim=True)` before computing the similarity matrix.

In [ ]:
# Demonstrate the alignment problem visually
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# BEFORE alignment
ax = axes[0]
ax.set_title('BEFORE Alignment Training', fontsize=14, fontweight='bold', color='#E74C3C')

# Image embeddings (clustered in one region)
img_pts = np.random.randn(5, 2) * 0.8 + np.array([3, 3])
txt_pts = np.random.randn(5, 2) * 0.8 + np.array([-2, -2])

labels = ['cat', 'dog', 'car', 'tree', 'house']
ax.scatter(img_pts[:, 0], img_pts[:, 1], s=150, c='#E74C3C', marker='s', 
           label='Image embeddings', zorder=5, edgecolors='white', linewidths=1.5)
ax.scatter(txt_pts[:, 0], txt_pts[:, 1], s=150, c='#3498DB', marker='o', 
           label='Text embeddings', zorder=5, edgecolors='white', linewidths=1.5)

for i, label in enumerate(labels):
    ax.annotate(f'img:{label}', img_pts[i], fontsize=9, 
                xytext=(5, 5), textcoords='offset points')
    ax.annotate(f'txt:{label}', txt_pts[i], fontsize=9, 
                xytext=(5, 5), textcoords='offset points')

ax.legend(fontsize=11)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')

# AFTER alignment (matching pairs are close)
ax = axes[1]
ax.set_title('AFTER Alignment Training (e.g. CLIP)', fontsize=14, fontweight='bold', color='#2ECC71')

centers = np.array([[-2, 3], [2, 3], [3, -1], [-3, -1], [0, -3]])
img_pts = centers + np.random.randn(5, 2) * 0.15
txt_pts = centers + np.random.randn(5, 2) * 0.15

ax.scatter(img_pts[:, 0], img_pts[:, 1], s=150, c='#E74C3C', marker='s', 
           label='Image embeddings', zorder=5, edgecolors='white', linewidths=1.5)
ax.scatter(txt_pts[:, 0], txt_pts[:, 1], s=150, c='#3498DB', marker='o', 
           label='Text embeddings', zorder=5, edgecolors='white', linewidths=1.5)

for i, label in enumerate(labels):
    ax.annotate(f'img:{label}', img_pts[i], fontsize=9, 
                xytext=(5, 5), textcoords='offset points')
    ax.annotate(f'txt:{label}', txt_pts[i], fontsize=9, 
                xytext=(5, 5), textcoords='offset points')
    # Draw line connecting matched pair
    ax.plot([img_pts[i, 0], txt_pts[i, 0]], [img_pts[i, 1], txt_pts[i, 1]],
            '--', color='#2ECC71', alpha=0.7, linewidth=2)

ax.legend(fontsize=11)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')

plt.tight_layout()
plt.savefig('../assets/alignment_problem.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. The Multimodal Model Zoo (Taxonomy)

Let's visualize the evolution and relationships between key models.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Evolution of Multimodal Models', fontsize=18, fontweight='bold', pad=20)

models = [
    # (x, y, name, year, color, type)
    (2, 8.5, 'ViT\n(2020)', '#E74C3C', 'Vision'),
    (5, 8.5, 'BERT\n(2018)', '#3498DB', 'Text'),
    (3.5, 7, 'CLIP\n(2021)', '#9B59B6', 'V+L'),
    (6.5, 7, 'ALIGN\n(2021)', '#9B59B6', 'V+L'),
    (1.5, 5.5, 'BLIP\n(2022)', '#F39C12', 'V+L'),
    (4.5, 5.5, 'Flamingo\n(2022)', '#F39C12', 'V+L'),
    (7.5, 5.5, 'BEiT-3\n(2022)', '#F39C12', 'V+L'),
    (3, 4, 'BLIP-2\n(2023)', '#2ECC71', 'V+L'),
    (6, 4, 'LLaVA\n(2023)', '#2ECC71', 'V+L'),
    (9, 4, 'InstructBLIP\n(2023)', '#2ECC71', 'V+L'),
    (4.5, 2.5, 'LLaVA-1.5\n(2024)', '#1ABC9C', 'V+L'),
    (7.5, 2.5, 'GPT-4V\n(2023)', '#1ABC9C', 'V+L'),
    
    (11, 8.5, 'Whisper\n(2022)', '#2ECC71', 'Audio'),
    (11, 7, 'ImageBind\n(2023)', '#E74C3C', 'Multi'),
    (13, 5.5, 'Gemini\n(2023)', '#1ABC9C', 'Multi'),
    (11, 4, 'Any-to-Any\n(2024)', '#34495E', 'Multi'),
]

for x, y, name, color, _ in models:
    draw_architecture_block(ax, x, y, 2.2, 0.8, name, color, fontsize=9)

# Key arrows showing evolution
connections = [
    (2, 8.0, 3.5, 7.5),
    (5, 8.0, 3.5, 7.5),
    (3.5, 6.5, 1.5, 6.0),
    (3.5, 6.5, 4.5, 6.0),
    (1.5, 5.0, 3, 4.5),
    (4.5, 5.0, 6, 4.5),
    (6, 3.5, 4.5, 3.0),
    (6, 3.5, 7.5, 3.0),
]
for x1, y1, x2, y2 in connections:
    draw_arrow(ax, (x1, y1), (x2, y2), color='gray')

# Legend
legend_items = [
    ('Vision Only', '#E74C3C'), ('Text Only', '#3498DB'),
    ('Vision+Language', '#9B59B6'), ('Multi-modal', '#1ABC9C')
]
for i, (label, color) in enumerate(legend_items):
    ax.add_patch(FancyBboxPatch((12.5, 2 - i*0.5), 0.3, 0.3, 
                                facecolor=color, alpha=0.8))
    ax.text(13, 2.15 - i*0.5, label, fontsize=10, va='center')

plt.tight_layout()
plt.savefig('../assets/model_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hands-On: Build a Mini-CLIP Model from Scratch

Let's build the **simplest possible multimodal model** — a miniature CLIP:

```
  Image ──→ CNN Encoder ──→ Projection ──→ L2 Normalize ──→ ┐
                                                              ├──→ Cosine Similarity
  Text  ──→ Transformer  ──→ Projection ──→ L2 Normalize ──→ ┘
```

The model learns: $\text{score}(I, T) = \frac{\mathbf{v}^\top \mathbf{t}}{\|\mathbf{v}\| \|\mathbf{t}\|} \cdot \frac{1}{\tau}$

### The Temperature Parameter $\tau$

CLIP scales cosine similarities by a **temperature** $\tau$ before applying softmax:

$$p_i = \frac{\exp(s_i / \tau)}{\sum_j \exp(s_j / \tau)}$$

| Temperature | Effect | Distribution Shape |
|-------------|--------|-------------------|
| **Low** $\tau = 0.01$ | Sharp peaks | Near one-hot — very confident |
| **Medium** $\tau = 0.07$ | Balanced (CLIP default) | Discriminative but not extreme |
| **High** $\tau = 1.0$ | Uniform | All similarities treated equally |

**Why $\tau = 0.07$?** Cosine similarities live in $[-1, 1]$, but softmax needs larger logit spreads to produce meaningful gradients. Dividing by 0.07 amplifies differences: $\text{sim}=0.9 \to 12.9$ vs $\text{sim}=0.1 \to 1.4$.

**Learnable temperature:** In our `MiniCLIP`, $\tau$ is a **learnable parameter** initialized to $\exp(\log(1/0.07)) = 1/0.07 \approx 14.3$ in logit space (CLIP stores `log(1/τ)` and divides logits by `exp(log_temp)`). The model learns the optimal sharpness during training.

In [ ]:
class SimpleImageEncoder(nn.Module):
    """Tiny CNN that maps an image to an embedding vector."""
    def __init__(self, embed_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.projection = nn.Linear(128, embed_dim)

    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.projection(x)


class SimpleTextEncoder(nn.Module):
    """Tiny transformer that maps token IDs to an embedding vector."""
    def __init__(self, vocab_size=1000, embed_dim=128, max_len=32):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.projection = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(x) + self.pos_embed(positions)
        x = self.transformer(x)
        x = x[:, 0]  # [CLS] token representation
        return self.projection(x)


class MiniCLIP(nn.Module):
    """Minimal CLIP-like model: align image and text embeddings."""
    def __init__(self, embed_dim=128):
        super().__init__()
        self.image_encoder = SimpleImageEncoder(embed_dim)
        self.text_encoder = SimpleTextEncoder(embed_dim=embed_dim)
        self.temperature = nn.Parameter(torch.ones(1) * 0.07)

    def forward(self, images, text_ids):
        img_emb = self.image_encoder(images)
        txt_emb = self.text_encoder(text_ids)

        # L2 normalize
        img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)

        # Cosine similarity scaled by temperature
        logits = img_emb @ txt_emb.T / self.temperature
        return logits, img_emb, txt_emb


# Detect best device (GPU on Colab, CPU locally)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Create model and inspect
model = MiniCLIP(embed_dim=128).to(device)
count_parameters(model)

In [ ]:
# Test with dummy data (move to same device as model)
batch_size = 4
images = torch.randn(batch_size, 3, 32, 32).to(device)
text_ids = torch.randint(0, 1000, (batch_size, 16)).to(device)

logits, img_emb, txt_emb = model(images, text_ids)

print(f"Image embeddings shape: {img_emb.shape}")   # [4, 128]
print(f"Text embeddings shape:  {txt_emb.shape}")    # [4, 128]
print(f"Similarity matrix shape: {logits.shape}")     # [4, 4]
print(f"Temperature τ = {model.temperature.item():.4f}")

# Visualize the similarity matrix (move to CPU for plotting)
fig = plot_attention_heatmap(
    logits.detach().cpu(),
    x_labels=[f'text_{i}' for i in range(4)],
    y_labels=[f'img_{i}' for i in range(4)],
    title='Image-Text Similarity Matrix (untrained — random)'
)
plt.savefig('../assets/similarity_untrained.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Interpretation:")
print("  • Diagonal = matching pairs (should be HIGH after training)")
print("  • Off-diagonal = non-matching pairs (should be LOW)")
print("  • Right now it's random — training with InfoNCE loss will fix this!")

In [ ]:
# ============================================================
#  Example 1: Train Mini-CLIP and Watch Alignment Emerge
# ============================================================
# Let's train our Mini-CLIP on synthetic data and watch
# the similarity matrix transform from random to diagonal!

import torch.nn.functional as F
from torch.optim import Adam

torch.manual_seed(42)
model = MiniCLIP(embed_dim=128).to(device)
optimizer = Adam(model.parameters(), lr=3e-4)

# Create synthetic paired data (colored shapes with matching labels)
N_SAMPLES = 200
images = torch.randn(N_SAMPLES, 3, 32, 32).to(device)
texts = torch.randint(0, 1000, (N_SAMPLES, 16)).to(device)

# Make matching pairs share structure: inject signal
for i in range(N_SAMPLES):
    class_id = i % 10
    images[i, class_id % 3, :, :] += 2.0  # color signal
    texts[i, 0] = class_id * 100           # token signal

def infonce_loss(logits):
    """Symmetric InfoNCE: image→text + text→image"""
    labels = torch.arange(logits.shape[0], device=logits.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2

# Training loop
losses = []
BATCH_SIZE = 32
print("Training Mini-CLIP...")
print(f"{'Epoch':>5} | {'Loss':>8} | {'Temp τ':>8} | {'Diag Mean':>10} | {'Off-diag Mean':>12}")
print("-" * 55)

for epoch in range(50):
    epoch_loss = 0
    for start in range(0, N_SAMPLES, BATCH_SIZE):
        end = min(start + BATCH_SIZE, N_SAMPLES)
        img_batch = images[start:end]
        txt_batch = texts[start:end]
        
        logits, _, _ = model(img_batch, txt_batch)
        loss = infonce_loss(logits)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / (N_SAMPLES // BATCH_SIZE)
    losses.append(avg_loss)
    
    if epoch % 10 == 0 or epoch == 49:
        with torch.no_grad():
            test_logits, _, _ = model(images[:8].to(device), texts[:8].to(device))
            sim = torch.softmax(test_logits, dim=-1)
            diag = sim.diag().mean().item()
            off_diag = (sim.sum() - sim.diag().sum()).item() / (64 - 8)
        print(f"{epoch:5d} | {avg_loss:8.4f} | {model.temperature.item():8.4f} | {diag:10.4f} | {off_diag:12.4f}")

print("\n✅ Training complete!")

In [ ]:
# ============================================================
#  Example 2: Before vs After — Visual Proof of Alignment
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Plot 1: Training loss curve
ax = axes[0]
ax.plot(losses, color='#E74C3C', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('InfoNCE Loss')
ax.set_title('Training Loss\n(should decrease)', fontsize=13, fontweight='bold')
ax.axhline(y=np.log(BATCH_SIZE), color='gray', linestyle='--', alpha=0.5, label=f'Random baseline (ln {BATCH_SIZE} = {np.log(BATCH_SIZE):.2f})')
ax.legend()

# Plot 2: Trained similarity matrix
ax = axes[1]
with torch.no_grad():
    test_logits, img_e, txt_e = model(images[:8].to(device), texts[:8].to(device))
    sim_matrix = (img_e @ txt_e.T).cpu().numpy()

im = ax.imshow(sim_matrix, cmap='YlOrRd', aspect='equal')
ax.set_title('Trained Similarity Matrix\n(diagonal = matching pairs)', fontsize=13, fontweight='bold')
ax.set_xlabel('Text samples')
ax.set_ylabel('Image samples')
for i in range(8):
    for j in range(8):
        color = 'white' if sim_matrix[i,j] > sim_matrix.mean() else 'black'
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

# Plot 3: Embedding space (2D PCA projection)
ax = axes[2]
with torch.no_grad():
    all_logits, all_img, all_txt = model(images[:50].to(device), texts[:50].to(device))
    img_np = all_img.cpu().numpy()
    txt_np = all_txt.cpu().numpy()

# Simple 2D projection using first 2 principal components
from numpy.linalg import svd
combined = np.vstack([img_np, txt_np])
combined -= combined.mean(axis=0)
U, S, Vt = svd(combined, full_matrices=False)
proj = combined @ Vt[:2].T

n = len(img_np)
classes = [i % 10 for i in range(n)]
scatter_colors = plt.cm.tab10(np.array(classes) / 10)

ax.scatter(proj[:n, 0], proj[:n, 1], c=scatter_colors, marker='s', s=60, alpha=0.8, label='Image', edgecolors='black', linewidths=0.5)
ax.scatter(proj[n:, 0], proj[n:, 1], c=scatter_colors, marker='o', s=60, alpha=0.8, label='Text', edgecolors='black', linewidths=0.5)

# Connect matching pairs for first 10
for i in range(10):
    ax.plot([proj[i, 0], proj[n+i, 0]], [proj[i, 1], proj[n+i, 1]], 
            'k--', alpha=0.3, linewidth=0.8)

ax.set_title('Embedding Space (PCA)\n(matching pairs cluster together)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')

plt.tight_layout()
plt.savefig('../assets/mini_clip_training.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 What we see:")
print("  • Loss decreased → model learned meaningful representations")
print("  • Similarity matrix has higher values on diagonal → matching pairs aligned")
print("  • PCA plot shows matching image-text pairs clustering together")

### Example 3: InfoNCE Loss — Complete Numerical Walkthrough

Let's trace InfoNCE with a **batch of 4** and temperature $\tau = 0.07$:

**Step 1 — Compute cosine similarities (after L2 normalization):**

| | text_0 | text_1 | text_2 | text_3 |
|--|--------|--------|--------|--------|
| **img_0** | **0.92** | 0.15 | -0.03 | 0.21 |
| **img_1** | 0.08 | **0.87** | 0.31 | -0.12 |
| **img_2** | -0.11 | 0.22 | **0.95** | 0.06 |
| **img_3** | 0.19 | -0.07 | 0.14 | **0.89** |

**Bold** = correct pairs (diagonal).

**Step 2 — Scale by temperature ($\div \tau = \div 0.07$):**

| | text_0 | text_1 | text_2 | text_3 |
|--|--------|--------|--------|--------|
| **img_0** | **13.14** | 2.14 | -0.43 | 3.00 |
| **img_1** | 1.14 | **12.43** | 4.43 | -1.71 |
| **img_2** | -1.57 | 3.14 | **13.57** | 0.86 |
| **img_3** | 2.71 | -1.00 | 2.00 | **12.71** |

**Step 3 — Row-wise softmax (image→text direction):**

For img_0: $p_0 = \frac{e^{13.14}}{e^{13.14} + e^{2.14} + e^{-0.43} + e^{3.00}} = \frac{507,739}{507,739 + 8.5 + 0.65 + 20.1} \approx 0.9994$

For img_1: $p_1 = \frac{e^{12.43}}{e^{1.14} + e^{12.43} + e^{4.43} + e^{-1.71}} \approx 0.9997$

**Step 4 — Cross-entropy loss:**

$$\mathcal{L}_{i2t} = -\frac{1}{4}[\log(0.9994) + \log(0.9997) + \log(0.9998) + \log(0.9995)]$$
$$= -\frac{1}{4}[-0.0006 + (-0.0003) + (-0.0002) + (-0.0005)] = 0.0004$$

This is an **excellent** loss — the model is very confident about correct pairs! Compare to random: $\mathcal{L}_{\text{random}} = \log(4) = 1.386$.

In [ ]:
# ============================================================
#  Example 3 (Code): Verify the InfoNCE Numerical Walkthrough
# ============================================================
# Let's reproduce the exact numbers from the walkthrough above

import torch.nn.functional as F

tau = 0.07

# Cosine similarity matrix (from our example)
S = torch.tensor([
    [0.92, 0.15, -0.03, 0.21],
    [0.08, 0.87,  0.31, -0.12],
    [-0.11, 0.22, 0.95,  0.06],
    [0.19, -0.07, 0.14,  0.89]
])

print("=" * 60)
print("InfoNCE Loss — Step-by-Step Verification")
print("=" * 60)

# Step 1: Scale by temperature
scaled = S / tau
print(f"\n1. Similarity matrix S (cosine):")
for i in range(4):
    row = " | ".join(f"{S[i,j]:6.2f}" for j in range(4))
    print(f"   img_{i}: [{row}]")

print(f"\n2. Scaled logits (S / τ = S / {tau}):")
for i in range(4):
    row = " | ".join(f"{scaled[i,j]:7.2f}" for j in range(4))
    print(f"   img_{i}: [{row}]")

# Step 2: Softmax (row-wise)
probs = torch.softmax(scaled, dim=-1)
print(f"\n3. Softmax probabilities (row-wise):")
for i in range(4):
    row = " | ".join(f"{probs[i,j]:7.4f}" for j in range(4))
    diag_val = probs[i,i].item()
    print(f"   img_{i}: [{row}]  ← P(correct) = {diag_val:.4f}")

# Step 3: Cross-entropy
labels = torch.arange(4)
loss_i2t = F.cross_entropy(scaled, labels)
loss_t2i = F.cross_entropy(scaled.T, labels)
total_loss = (loss_i2t + loss_t2i) / 2

print(f"\n4. Loss calculation:")
print(f"   L_i2t = {loss_i2t.item():.6f}")
print(f"   L_t2i = {loss_t2i.item():.6f}")
print(f"   L_total = (L_i2t + L_t2i) / 2 = {total_loss.item():.6f}")
print(f"   Random baseline = ln(4) = {np.log(4):.4f}")
print(f"\n   ✅ Loss is {total_loss.item():.4f} vs random {np.log(4):.4f}")
print(f"   → The model is {np.log(4)/total_loss.item():.0f}x better than random!")

# Bonus: What happens with a BAD model?
print("\n" + "=" * 60)
print("Comparison: What if the model is confused?")
print("=" * 60)
S_bad = torch.tensor([
    [0.25, 0.30, 0.22, 0.28],
    [0.27, 0.24, 0.29, 0.26],
    [0.23, 0.28, 0.25, 0.30],
    [0.29, 0.26, 0.27, 0.24]
])
scaled_bad = S_bad / tau
loss_bad = F.cross_entropy(scaled_bad, labels)
print(f"   Loss (confused model) = {loss_bad.item():.4f}")
print(f"   Loss (aligned model)  = {loss_i2t.item():.4f}")
print(f"   → Aligned model has {loss_bad.item()/loss_i2t.item():.0f}x lower loss!")

### Example 4: Real-World Application — Zero-Shot Image Classification

In practice, CLIP enables **zero-shot classification** without any task-specific training. Here's how it works:

**Scenario:** You have a medical imaging system that needs to classify X-rays into categories, but you have NO labeled training data for this specific task.

**CLIP's Solution:**
1. Encode each X-ray image → image embedding
2. Encode text prompts like "an X-ray showing pneumonia" → text embeddings  
3. Pick the text with highest similarity → that's the predicted class

No training needed! Just a good set of text prompts (called "prompt engineering").

In [ ]:
# ============================================================
#  Example 4: Zero-Shot Classification with Mini-CLIP
# ============================================================
# Simulate zero-shot classification: classify images using text descriptions

torch.manual_seed(42)

# Create 5 "classes" of synthetic images with distinct visual patterns
class_names = ["red circle", "blue square", "green triangle", "yellow star", "purple diamond"]
N_PER_CLASS = 10

test_images = torch.randn(50, 3, 32, 32).to(device)
true_labels = []

# Inject class-specific signals
for cls_id in range(5):
    for j in range(N_PER_CLASS):
        idx = cls_id * N_PER_CLASS + j
        test_images[idx, cls_id % 3, :, :] += 3.0  # strong color signal
        true_labels.append(cls_id)

# Create text embeddings for each class description
class_texts = torch.zeros(5, 16, dtype=torch.long).to(device)
for c in range(5):
    class_texts[c, 0] = c * 100  # matching token signal from training

# Zero-shot classification
model.eval()
with torch.no_grad():
    # Encode all class descriptions once
    _, _, class_embs = model(test_images[:5].to(device), class_texts)
    class_embs = class_embs / class_embs.norm(dim=-1, keepdim=True)
    
    # Classify each test image
    predictions = []
    confidences = []
    for i in range(50):
        _, img_emb, _ = model(test_images[i:i+1], class_texts[:1])  # dummy text
        img_emb = model.image_encoder(test_images[i:i+1])
        img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        
        sims = (img_emb @ class_embs.T).squeeze()
        pred = sims.argmax().item()
        conf = torch.softmax(sims / 0.07, dim=0)[pred].item()
        predictions.append(pred)
        confidences.append(conf)

# Calculate accuracy
correct = sum(p == t for p, t in zip(predictions, true_labels))
accuracy = correct / len(true_labels) * 100

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix
ax = axes[0]
conf_matrix = np.zeros((5, 5), dtype=int)
for t, p in zip(true_labels, predictions):
    conf_matrix[t][p] += 1
im = ax.imshow(conf_matrix, cmap='Blues')
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(class_names, fontsize=9)
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(conf_matrix[i][j]), ha='center', va='center', fontsize=12,
                color='white' if conf_matrix[i][j] > 5 else 'black')
ax.set_title(f'Zero-Shot Classification\nAccuracy: {accuracy:.1f}%', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.colorbar(im, ax=ax, shrink=0.8)

# Confidence distribution
ax = axes[1]
ax.hist(confidences, bins=20, color='#2ECC71', alpha=0.7, edgecolor='white')
ax.axvline(x=np.mean(confidences), color='red', linestyle='--', label=f'Mean conf: {np.mean(confidences):.3f}')
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../assets/zero_shot_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🎯 Zero-Shot Results: {accuracy:.1f}% accuracy with NO task-specific training")
print(f"   Mean confidence: {np.mean(confidences):.3f}")
print(f"   This is the power of contrastive pretraining!")

### Example 5: Real-World Multimodal Applications

Here are concrete industry applications using the concepts from this notebook:

| Application | Modalities | Model | How It Works |
|------------|-----------|-------|--------------|
| **Medical Image Search** | X-ray + Text | CLIP-based | Doctor types "fracture in left femur" → system retrieves matching X-rays from database |
| **Autonomous Driving** | Camera + LiDAR + Text | CLIP + 3D encoder | "Stop sign ahead" understanding from visual + depth data |
| **E-commerce Search** | Product Image + Description | OpenCLIP | Customer uploads photo → finds similar products with matching descriptions |
| **Content Moderation** | Image + Caption | CLIP | Zero-shot detection: "this image contains violence" similarity scoring |
| **Accessibility** | Image → Text | BLIP/LLaVA | Automatic alt-text generation for visually impaired users |
| **Document Understanding** | Scanned Doc + OCR Text | LayoutLM | Combine visual layout with extracted text for form parsing |

**Key Pattern:** All these applications follow the same recipe we built:
1. Encode each modality with a specialized encoder
2. Project into shared embedding space  
3. Use cosine similarity for matching/retrieval
4. Fine-tune on domain data if needed (see Module 04)

### InfoNCE Loss — Preview

The similarity matrix you just saw is trained with **InfoNCE** (Noise Contrastive Estimation) loss — the heart of contrastive learning:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log \frac{\exp(\text{sim}(\mathbf{v}_i, \mathbf{t}_i) / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(\mathbf{v}_i, \mathbf{t}_j) / \tau)}$$

**Reading the formula:**
- **Numerator:** similarity of the **correct** image-text pair $(i, i)$
- **Denominator:** similarity of image $i$ against **all** text candidates in the batch
- The loss pushes the diagonal (matching pairs) **up** and off-diagonal (non-matching) **down**

This is exactly softmax cross-entropy where each image must "pick" its correct text caption from $N$ options. With batch size 32,768 (as in original CLIP), each sample has 32,767 negative examples — that's why CLIP needs large batches!

We explore InfoNCE in full mathematical depth in **Module 03: Training Strategies**.

## Key Takeaways

1. **Multimodal = multiple input types** sharing a learned representation space
2. Each modality needs its own **encoder** (ViT for images, BERT for text, etc.)
3. All encoders output vectors in the **same dimension** $\mathbf{h} \in \mathbb{R}^D$ — this enables fusion
4. The core challenge is **alignment**: $\text{sim}(\mathbf{v}_i, \mathbf{t}_i) \gg \text{sim}(\mathbf{v}_i, \mathbf{t}_j)$ for matching pair $i$
5. **Contrastive learning** (CLIP-style InfoNCE) is the most popular alignment method
6. Modern models (LLaVA, GPT-4V) extend this by connecting vision encoders to LLMs

---

## What's Next

In the next notebook, we'll dive deep into the **encoders** — building a Vision Transformer (ViT) and Text Encoder from scratch, with full math.

👉 **[02_modality_encoders.ipynb](./02_modality_encoders.ipynb)** — Build ViT and Text Encoder from raw PyTorch

---

## 📚 References & Further Reading

### Papers
- **Learning Transferable Visual Models From Natural Language Supervision (CLIP)** — Radford et al., 2021 — [arXiv:2103.00020](https://arxiv.org/abs/2103.00020) — Foundation of vision-language alignment
- **Visual Instruction Tuning (LLaVA)** — Liu et al., 2023 — [arXiv:2304.08485](https://arxiv.org/abs/2304.08485) — Connecting vision encoders to LLMs
- **Flamingo: a Visual Language Model for Few-Shot Learning** — Alayrac et al., 2022 — [arXiv:2204.14198](https://arxiv.org/abs/2204.14198) — Few-shot multimodal reasoning
- **ImageBind: One Embedding Space To Bind Them All** — Girdhar et al., 2023 — [arXiv:2305.05665](https://arxiv.org/abs/2305.05665) — 6-modality unified embeddings
- **Robust Speech Recognition via Large-Scale Weak Supervision (Whisper)** — Radford et al., 2022 — [arXiv:2212.04356](https://arxiv.org/abs/2212.04356) — Audio-to-text foundation model
- **Representation Learning with Contrastive Predictive Coding (CPC)** — van den Oord et al., 2018 — [arXiv:1807.03748](https://arxiv.org/abs/1807.03748) — InfoNCE loss origin

### Blog Posts & Cheat Sheets
- 🔗 [The Illustrated CLIP](https://blog.roboflow.com/openai-clip/) — Roboflow — Visual guide to CLIP architecture
- 🔗 [Multimodal Deep Learning Survey](https://arxiv.org/abs/2301.04856) — Comprehensive 2023 survey paper
- 🔗 [Lilian Weng: Contrastive Representation Learning](https://lilianweng.github.io/posts/2021-05-31-contrastive/) — Excellent overview of contrastive methods
- 🔗 [OpenAI CLIP Blog](https://openai.com/research/clip) — Original CLIP announcement with demos
- 🔗 [Hugging Face Multimodal Models](https://huggingface.co/docs/transformers/model_doc/clip) — CLIP implementation and usage guide
- 🔗 [Google AI Blog: Gemini](https://blog.google/technology/ai/google-gemini-ai/) — Native multimodal architecture